# 03 - Introduction to Agents with Google ADK

In the first section, we built an **AI Travel Assistant** — a basic LLM app that can answer travel-related questions.

### Answers, but no actions
The initial assistant can generate helpful responses, but it cannot act. It is limited to the prompt and its training data. It cannot check the weather, search for flights, or convert currencies.

### Moving towards agents
To make the assistant more useful, we need it to go beyond text generation. We want it to:

- Use external tools
- Decide when a tool is needed
- Combine tool results into one useful answer

In this notebook we use **Google Agent Development Kit (ADK)**. ADK gives us an `Agent`, a `Runner`, and session management so the model can call Python tools as part of an agent turn.


## Setup

Before running the below cell, ensure you have:

1. Authenticated with your WIF-backed credentials, for example through `gcloud auth application-default login` or your workshop's WIF setup
2. Set your GCP project and location below

This notebook does **not** use API keys. ADK is configured to use Vertex AI and Application Default Credentials.


In [ ]:
import json
import os
import uuid
from dataclasses import dataclass
from pathlib import Path

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import GenerateContentConfig, Content, Part

In [ ]:
# Select the model
MODEL_NAME = "gemini-2.5-flash"
APP_NAME = "Travel-Assistant"
USER_ID = "workshop-user"

# Set a default instruction for the agent
SYSTEM_MESSAGE = """
You are a helpful travel assistant.
You have access to tools. Decide whether you need to use a tool.
- If needed, use it
- Otherwise, answer directly
"""


@dataclass
class AgentResponse:
    text: str
    function_calls: list[dict]


async def ask_agent(
    prompt: str,
    tools: list | None = None,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 1.0,
) -> AgentResponse:
    """Run one ADK agent turn and return the final text plus tool calls."""

    config = GenerateContentConfig(
        temperature=temperature,                # <-- Controls randomness: 0=deterministic, 1=creative, 2=very random
        top_p=top_p,                            # <-- Nucleus sampling: considers tokens with cumulative probability up to this value
        top_k=top_k,                            # <-- Limits sampling to the top K most likely tokens at each step
    )

    agent = Agent(
        name="travel_assistant",
        model=MODEL_NAME,
        instruction=system_instruction,
        tools=tools or [],
        generate_content_config=config
    )

    session_service = InMemorySessionService()
    session_id = f"session-{uuid.uuid4().hex}"
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        app_name=APP_NAME,
        agent=agent,
        session_service=session_service,
    )

    message = Content(
        role="user",
        parts=[Part(text=prompt)],
    )

    final_text = ""
    function_calls = []

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        for call in event.get_function_calls():
            function_calls.append({"name": call.name, "args": dict(call.args or {})})

        if event.is_final_response() and event.content and event.content.parts:
            text_parts = [part.text for part in event.content.parts if part.text]
            if text_parts:
                final_text = "\n".join(text_parts)

    return AgentResponse(text=final_text, function_calls=function_calls)


## First tool call

**What is a tool?**

A tool is just a python function the LLM can call for a specific task.

For this workshop, we use tools that read local mock data instead of calling real weather, flight, or exchange-rate APIs.

#### Load mock tool data

These files act like tiny fake APIs. They make the workshop repeatable because the data does not change.

In [ ]:
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

with open(DATA_DIR / "mock_weather.json", "r", encoding="utf-8") as file:
    weather_data = json.load(file)

with open(DATA_DIR / "mock_flights.json", "r", encoding="utf-8") as file:
    flight_data = json.load(file)

print("Mock data loaded.")


#### Create mock tools

Each tool is a normal Python function. The model does not call these directly yet.

In [ ]:
def get_weather(location: str) -> dict:
    """Returns the current weather for a location.
    Args:
        location: The city name, for example: Lisbon
    """
    print(f"🔧 TOOL CALLED: get_weather(location={location})")
    return weather_data.get(location,{"error": f"No mock weather found for {location}."})


def search_flights(origin: str, destination: str) -> list:
    """Return mock flights for an origin-destination pair."
     Args:
        origin: The departure city, for example: Lisbon
        destination: The arrival city, for example: Paris
    """
    print(f"🔧 TOOL CALLED: search_flights(origin={origin}, destination={destination})")
    route = f"{origin}-{destination}"
    return flight_data.get(route, [])


def your_tool():
    """Define your own tool here!"""
    pass


In [ ]:
prompt = """
I am visiting Lisbon this weekend. What should I pack?
"""

response = await ask_agent(
    prompt,
    tools=[get_weather],
)

print(f"Response:{response.text}")
print(f"Tool calls:{json.dumps(response.function_calls, indent=2)}")


## Multi-tool workflow

Now we ask a question that may require more than one tool.

The assistant needs to:
- Search for a flight
- Check company travel policy
- Possibly convert currency or explain cost


This is closer to an agent workflow.

The assistant may need to combine information from multiple tools before giving an answer.


In [ ]:
prompt = """
I am planning a weekend work trip to Lisbon.
Please help me decide what to pack, and whether a flight is available.
I will be departing from Amsterdam.
"""

response = await ask_agent(
    prompt,
    tools=[get_weather, search_flights],
)

print("-----"*10)
print(f"Response:\n\n{response.text}")
print("-----"*10)
print(f"Tool calls:{json.dumps(response.function_calls, indent=2)}")

## Exercise: Add Your Own Tool

Now it's your turn to extend the agent!

Create a new tool function. Your tool should take input parameters and return a mock result.

Make sure to update the prompt* so that the user's question clearly requires your tool (e.g., "How much should I budget per day in Paris for a student?").

**Example Ideas:**
- `estimate_daily_budget(city, style)`
- `exchange_rate_calculator(from_currency, to_currency)`
- `suggest_airport_transfer(city)`

Try it out and see how the LLM responds when you provide tool results!